In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilace s pass managery

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vytvořen s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Doporučený způsob transpilace obvodu je vytvořit staged pass manager a poté zavolat jeho metodu `run` s obvodem jako vstupem. Tato stránka vysvětluje, jak tímto způsobem transpilovat kvantové obvody.
## Co je (staged) pass manager?
V kontextu Qiskit SDK se transpilací rozumí proces transformace vstupního obvodu do podoby vhodné pro spuštění na kvantovém zařízení. Transpilace obvykle probíhá v posloupnosti kroků nazývaných transpiler passes. Obvod je postupně zpracováván jednotlivými transpiler passes, přičemž výstup jednoho passu se stává vstupem dalšího. Například jeden pass může projít obvodem a sloučit všechny po sobě jdoucí sekvence jednoqubitových Gate, a následující pass může tyto Gate syntetizovat do základní sady cílového zařízení. Transpiler passes zahrnuté v Qiskitu jsou umístěny v modulu [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Pass manager je objekt, který uchovává seznam transpiler passes a může je spouštět na obvodu. Pass manager vytvoříš inicializací [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) se seznamem transpiler passes. Pro spuštění transpilace na obvodu zavolej metodu [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) s obvodem jako vstupem.

Staged pass manager je speciální druh pass manageru, který představuje vyšší úroveň abstrakce než běžný pass manager. Zatímco běžný pass manager se skládá z několika transpiler passes, staged pass manager se skládá z několika *pass managerů*. Jde o užitečnou abstrakci, protože transpilace obvykle probíhá v diskrétních fázích, jak je popsáno v části [Fáze Transpileru](transpiler-stages), přičemž každou fázi reprezentuje jeden pass manager. Staged pass managery jsou reprezentovány třídou [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). Zbývající část této stránky popisuje, jak (staged) pass managery vytvářet a přizpůsobovat.
## Vygenerování přednastavených staged pass managerů
Pro vytvoření přednastavených staged pass managerů s rozumnými výchozími hodnotami použij funkci [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Chceš-li transpilovat obvod nebo seznam obvodů pomocí pass manageru, předej obvod nebo seznam obvodů metodě `run`. Ukážeme si to na dvouqubitovém obvodu, který se skládá z Hadamardovy brány následované dvěma sousedními CX Gate:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Popis možných argumentů funkce `generate_preset_pass_manager` najdeš v části [Výchozí nastavení transpilace a možnosti konfigurace](defaults-and-configuration-options). Argumenty funkce `generate_preset_pass_manager` odpovídají argumentům funkce [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Máš potíže se zapamatováním detailů pass managerů? Zkus se zeptat Qiskit Code Assistant." />

Pokud přednastavené pass managery nesplňují tvé potřeby, můžeš transpilaci přizpůsobit vytvořením vlastních (staged) pass managerů nebo dokonce vlastních transpilačních passů. Zbývající část této stránky popisuje, jak vytvářet pass managery. Pokyny k vytváření transpilačních passů najdeš v části [Napište si vlastní transpiler pass](custom-transpiler-pass).

## Vytvoření vlastního pass manageru

Modul [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) obsahuje mnoho transpiler passes, které lze použít k vytváření pass managerů. Pass manager vytvoříš inicializací `PassManager` se seznamem passů. Například následující kód vytvoří transpiler pass, který sloučí sousední dvouqubitové Gate a poté je syntetizuje do základny [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) a [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Pro ukázku tohoto pass manageru v praxi ho otestuj na dvouqubitovém obvodu, který se skládá z Hadamardovy brány následované dvěma sousedními CX Gate:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Chceš-li spustit pass manager na obvodu, zavolej metodu `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Pokročilejší příklad ukazující, jak vytvořit pass manager pro implementaci techniky potlačení chyb známé jako dynamical decoupling, najdeš v části [Vytvoření pass manageru pro dynamical decoupling](dynamical-decoupling-pass-manager).

## Vytvoření staged pass manageru

`StagedPassManager` je pass manager skládající se z jednotlivých fází, přičemž každá fáze je definována instancí `PassManager`. `StagedPassManager` vytvoříš zadáním požadovaných fází. Například následující kód vytvoří staged pass manager se dvěma fázemi, `init` a `translation`. Fáze `translation` je definována pass managerem vytvořeným dříve.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Počet fází, které lze vložit do staged pass manageru, není omezen.

Dalším užitečným způsobem, jak vytvořit staged pass manager, je začít s přednastavenými staged pass managery a poté vyměnit některé z fází. Například následující kód vygeneruje přednastavený pass manager s úrovní optimalizace 3 a poté specifikuje vlastní fázi `pre_layout`.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[Funkce generátorů fází](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) mohou být užitečné při sestavování vlastních pass managerů.
Generují fáze, které poskytují běžnou funkčnost využívanou v mnoha pass managerech.
Například [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) lze použít ke generování fáze pro „vložení" vybraného počátečního `Layout` z layout passu na zadané cílové zařízení.

## Další kroky
> **Tip:** - [Napište si vlastní transpiler pass](custom-transpiler-pass).
>     - [Vytvoření pass manageru pro dynamical decoupling](dynamical-decoupling-pass-manager).
>     - Chceš-li se dozvědět více o funkci `generate_preset_passmanager`, viz téma [Výchozí nastavení transpilace a možnosti konfigurace](defaults-and-configuration-options).
>     - Vyzkoušej průvodce [Porovnání nastavení Transpileru](/guides/circuit-transpilation-settings).
>     - Prostuduj si [dokumentaci API Transpileru.](https://docs.quantum.ibm.com/api/qiskit/transpiler)